In [1]:
# ============================================================
# S5: CONTEXT-WINDOW STUDENT + KD (lambda=0.5) +
#     ARTIFACT-WEIGHTED LOSS
#
# Motivation: S1-S4 all gave the student only a SINGLE center
# epoch of Zmax+E4 as input. The teacher, by contrast, sees a
# +-7 epoch context window (WINDOW=15). N1 is a transitional
# stage that is defined largely by surrounding-epoch context,
# so a single-epoch student is architecturally handicapped for
# N1 specifically -- consistent with N1 being the weakest class
# across every S1-S4 result so far.
#
# This script keeps the best loss configuration found so far
# (artifact-weighted CE + KD, lambda=0.5) and changes ONLY the
# input/architecture: each modality now receives a +-CONTEXT
# window of epochs, encoded per-epoch and then aggregated across
# the window with an inter-epoch BiGRU (mirroring the teacher's
# two-stage intra/inter-epoch design, but lightweight).
#
#   Zmax window (W,2,1920) -> per-epoch ZmaxEEGEncoder -> (W,d)
#                           -> inter-epoch BiGRU -> center repr
#   E4   window (W,3,1920) -> per-epoch E4Encoder      -> (W,d)
#                           -> inter-epoch BiGRU -> center repr
#   concat -> classifier
#
# Compare against S4(lambda=0.5) ensemble (single-epoch input,
# F1=0.5277 pre-ensemble) to isolate the effect of adding
# temporal context to the student.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
PSG_EEG_PATH      = r"D:\22\AA\preprocess\preprocessed_FFinal"
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"
TEACHER_CKPT_DIR  = r"D:\22\AA\evaluation\multiscale_plain_c7_88%"
EVAL_PATH         = r"D:\22\AA\evaluation\teacher-student\s5_context_window_kd"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
CONTEXT     = 7                    # same context size as the teacher, for direct comparability
WINDOW      = 2 * CONTEXT + 1       # 15
N_EPOCHS    = 30
BATCH_SIZE  = 32                    # reduced from 64: window inputs are ~15x larger per sample
SEEDS         = [42, 123, 256, 789, 999]
TEACHER_SEEDS = [42, 123, 256, 789, 999]

T_D_MODEL = 128
T_DROPOUT = 0.4
T_N_HEADS = 4

S_D_MODEL = 96
S_DROPOUT = 0.3

KD_LAMBDA      = 0.5     # best value found in the S2/S4 sweep
KD_TEMPERATURE = 3.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"S5: CONTEXT-WINDOW student (+-{CONTEXT} epochs) + KD + artifact-weighted CE")
print(f"KD lambda     : {KD_LAMBDA}   Temperature: {KD_TEMPERATURE}")
print(f"Teacher       : MultiScaleSleepNetPlain, 5-seed ensemble (frozen)")
print(f"Output        : {EVAL_PATH}")


# ============================================================
# ============  TEACHER ARCHITECTURE (frozen)  ================
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=T_D_MODEL, dropout=T_DROPOUT):
        super().__init__()
        mid = d_model // 4

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)
        self.medium = time_branch(50)
        self.large  = time_branch(100)

        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)
        feat = self.se(feat)
        return self.proj(feat)


class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=T_D_MODEL, n_heads=T_N_HEADS, dropout=T_DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            d_model, d_model // 2, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)
        attn_out = self.transformer(x)
        return self.norm2(x + attn_out)


class MultiScaleSleepNetPlain(nn.Module):
    """Teacher architecture -- identical to your trained checkpoints."""
    def __init__(self, in_ch=3, d_model=T_D_MODEL, n_layers=2,
                 dropout=T_DROPOUT, n_classes=5, context=CONTEXT):
        super().__init__()
        self.context = context
        self.d_model = d_model
        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)
        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.inter_pos = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes),
        )

    def forward(self, x):
        B, W, C, T = x.shape
        cnn_out = self.cnn(x.view(B * W, C, T)).permute(0, 2, 1)
        intra = self.intra_blocks(cnn_out)
        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)
        inter = self.inter_blocks(epoch_feat + self.inter_pos)
        center = inter[:, self.context, :]
        return self.classifier(center)


# ============================================================
# ========  STUDENT: PER-EPOCH ENCODER (same as S4)  ===========
# Applied to EACH epoch in the window via reshape, then an
# inter-epoch BiGRU aggregates across the window.
# ============================================================
class ZmaxEEGEncoder(nn.Module):
    """Per-epoch encoder -- unchanged from S4."""
    def __init__(self, in_ch=2, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        mid = d_model // 2

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2), nn.Dropout(dropout)
            )
        self.small, self.large = branch(15), branch(60)
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 1920)
            L_s, L_l = self.small(dummy).shape[2], self.large(dummy).shape[2]
        target_L = min(L_s, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.proj = nn.Sequential(nn.Conv1d(2 * mid, d_model, kernel_size=1),
                                   nn.BatchNorm1d(d_model), nn.GELU())
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (N, 2, 1920) where N = B*W (flattened batch*window)
        fs, fl = self.pool_s(self.small(x)), self.pool_l(self.large(x))
        feat = self.proj(torch.cat([fs, fl], dim=1)).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))   # (N, d_model)


class E4Encoder(nn.Module):
    """Per-epoch encoder -- unchanged from S4."""
    def __init__(self, in_ch=3, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, d_model // 2, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(d_model // 2), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model // 2, d_model, kernel_size=8, padding=4),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        feat = self.conv(x).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class InterEpochAggregator(nn.Module):
    """
    Aggregates a (B, W, d_model) sequence of per-epoch embeddings
    across the window using a BiGRU, then returns the CENTER
    epoch's contextualized representation -- this is what gives
    the student access to surrounding-epoch information, which
    single-epoch S1-S4 completely lacked.
    """
    def __init__(self, d_model=S_D_MODEL, context=CONTEXT):
        super().__init__()
        self.context = context
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, W, d_model)
        gru_out, _ = self.gru(x)              # (B, W, d_model)
        contextualized = self.norm(x + gru_out)
        return contextualized[:, self.context, :]   # (B, d_model) -- center epoch


class ContextStudentModel(nn.Module):
    """
    Context-window student: each modality gets a per-epoch
    encoder applied across the window, then an inter-epoch BiGRU
    aggregator extracts a context-aware representation of the
    CENTER epoch. Fusion is simple concat (best-performing so far).
    """
    def __init__(self, d_model=S_D_MODEL, n_classes=5, dropout=S_DROPOUT, context=CONTEXT):
        super().__init__()
        self.context = context
        self.zmax_enc = ZmaxEEGEncoder(in_ch=2, d_model=d_model, dropout=dropout)
        self.e4_enc   = E4Encoder(in_ch=3, d_model=d_model, dropout=dropout)
        self.zmax_agg = InterEpochAggregator(d_model=d_model, context=context)
        self.e4_agg   = InterEpochAggregator(d_model=d_model, context=context)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, zmax_x, e4_x):
        # zmax_x: (B, W, 2, 1920)   e4_x: (B, W, 3, 1920)
        B, W, Cz, Tz = zmax_x.shape
        _, _, Ce, Te = e4_x.shape

        z_flat = self.zmax_enc(zmax_x.reshape(B * W, Cz, Tz)).view(B, W, -1)
        e_flat = self.e4_enc(e4_x.reshape(B * W, Ce, Te)).view(B, W, -1)

        z_ctx = self.zmax_agg(z_flat)   # (B, d_model) -- center epoch, context-aware
        e_ctx = self.e4_agg(e_flat)     # (B, d_model)

        fused = torch.cat([z_ctx, e_ctx], dim=1)
        logits = self.classifier(fused)
        return logits


# ============================================================
# COMBINED DATASET (student now returns a WINDOW of Zmax+E4 epochs)
# ============================================================
class KDContextDataset(Dataset):
    def __init__(self, subject_list, psg_path, student_path, context=CONTEXT):
        self.context = context
        self.data = []
        self.index = []

        skipped_mismatch = 0
        for sub in subject_list:
            psg_fp = os.path.join(psg_path, f"{sub}.npz")
            stu_fp = os.path.join(student_path, f"{sub}.npz")
            if not (os.path.exists(psg_fp) and os.path.exists(stu_fp)):
                continue

            with np.load(psg_fp) as d:
                eeg = d['eeg'][:, [0, 1], :]
                eog = d['eog'][:, [0], :]
                psg_signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                psg_labels = d['labels'].copy()

            with np.load(stu_fp) as d:
                zmax_arr = d['zmax_eeg']
                e4_arr   = d['e4']
                stu_labels = d['labels'].copy()
                art_w = d['artifact_weight']

            n = min(len(psg_labels), len(stu_labels), zmax_arr.shape[0], e4_arr.shape[0])
            if abs(len(psg_labels) - len(stu_labels)) > 5:
                skipped_mismatch += 1

            sub_idx = len(self.data)
            self.data.append((psg_signal, psg_labels, zmax_arr, e4_arr, art_w, stu_labels, n))
            for i in range(n):
                self.index.append((sub_idx, i, n))

        print(f"  Subjects loaded: {len(self.data)}  (large mismatches: {skipped_mismatch})")
        print(f"  Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n = self.index[idx]
        psg_signal, psg_labels, zmax_arr, e4_arr, art_w, stu_labels, _ = self.data[sub_idx]

        # Teacher window (PSG)
        psg_window = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            psg_window.append(psg_signal[ei])
        psg_x = np.stack(psg_window, axis=0)

        # Student window (Zmax + E4) -- same padding-at-boundary convention
        zmax_window, e4_window = [], []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            zmax_window.append(zmax_arr[ei])
            e4_window.append(e4_arr[ei])
        zmax_x = np.stack(zmax_window, axis=0)   # (W, 2, 1920)
        e4_x   = np.stack(e4_window, axis=0)     # (W, 3, 1920)

        aw = float(art_w[center_i]) if center_i < len(art_w) else 1.0
        y  = int(stu_labels[center_i]) if center_i < len(stu_labels) else int(psg_labels[center_i])

        return (
            torch.FloatTensor(psg_x),
            torch.FloatTensor(zmax_x),
            torch.FloatTensor(e4_x),
            torch.tensor(aw, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
        )


# ============================================================
# LOAD SPLIT + BUILD DATASETS
# ============================================================
_train_path = os.path.join(PSG_EEG_PATH, "_train_subs.npy")
_test_path  = os.path.join(PSG_EEG_PATH, "_test_subs.npy")
TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()
print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Test:{len(TEST_SUBS)}")

print("\nBuilding context-window KD datasets (this uses more RAM than S1-S4)...")
train_ds = KDContextDataset(TRAIN_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print()
test_ds  = KDContextDataset(TEST_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print("Datasets ready.")


# ============================================================
# LOAD FROZEN TEACHER ENSEMBLE
# ============================================================
print(f"\nLoading {len(TEACHER_SEEDS)} frozen teacher checkpoints...")
teachers = []
for seed in TEACHER_SEEDS:
    ckpt = os.path.join(TEACHER_CKPT_DIR, f"best_seed{seed}.pt")
    if not os.path.exists(ckpt):
        print(f"  MISSING: {ckpt}")
        continue
    m = MultiScaleSleepNetPlain(in_ch=3).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    teachers.append(m)
print(f"Loaded {len(teachers)} teacher models (frozen).")


@torch.no_grad()
def teacher_soft_targets(psg_x, temperature=KD_TEMPERATURE):
    probs_sum = None
    for m in teachers:
        logits = m(psg_x)
        probs = F.softmax(logits / temperature, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(teachers)


# ============================================================
# CLASS WEIGHTS
# ============================================================
all_train_labels = []
for _, _, _, _, _, labs, _ in train_ds.data:
    all_train_labels.extend(labs.tolist())
label_counts = np.array([Counter(all_train_labels).get(i, 1) for i in range(5)], dtype=np.float32)
cw = torch.FloatTensor(label_counts.sum() / (5 * label_counts)).to(device)
print(f"\nClass weights: {dict(zip(LABEL_NAMES, cw.cpu().numpy().round(3)))}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def artifact_weighted_kd_loss_fn(student_logits, teacher_probs, labels, artifact_weight,
                                  temperature, lam, class_weights):
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    kd = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * (temperature ** 2)

    ce_per_sample = F.cross_entropy(student_logits, labels, weight=class_weights, reduction='none')
    weighted_ce = ce_per_sample * artifact_weight
    valid = artifact_weight > 0
    ce = weighted_ce[valid].mean() if valid.sum() > 0 else weighted_ce.mean()

    total = ce + lam * kd
    return total, kd.item(), ce.item()


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = total_kd = total_ce = 0
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, aw, y in loader:
        psg_x, zmax_x, e4_x, aw, y = (
            psg_x.to(device), zmax_x.to(device), e4_x.to(device),
            aw.to(device), y.to(device)
        )
        optimizer.zero_grad()
        teacher_probs = teacher_soft_targets(psg_x)
        student_logits = model(zmax_x, e4_x)
        loss, kd_val, ce_val = artifact_weighted_kd_loss_fn(
            student_logits, teacher_probs, y, aw, KD_TEMPERATURE, KD_LAMBDA, cw
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item(); total_kd += kd_val; total_ce += ce_val
        preds.extend(student_logits.argmax(1).cpu().numpy())
        labs_all.extend(y.cpu().numpy())

    n = len(loader)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    return total_loss / n, total_kd / n, total_ce / n, acc, f1


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, aw, y in loader:
        zmax_x, e4_x = zmax_x.to(device), e4_x.to(device)
        logits = model(zmax_x, e4_x)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs_all.extend(y.numpy())
    preds, labs_all = np.array(preds), np.array(labs_all)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs_all, preds)
    per_cls = f1_score(labs_all, preds, average=None, zero_division=0)
    return acc, f1, kappa, per_cls


# ============================================================
# TRAINING LOOP
# ============================================================
csv_path = os.path.join(EVAL_PATH, "summary.csv")
fields = ["seed", "acc", "f1_macro", "kappa", "f1_Wake", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fields).writeheader()

all_results = []
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSTUDENT SEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=0, generator=torch.Generator().manual_seed(seed))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = ContextStudentModel().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Student parameters: {n_params:,}  (teacher frozen, not counted)")

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4, betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_f1 = 0.0
    best_path = os.path.join(EVAL_PATH, f"best_student_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_kd, tr_ce, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler)
        vl_acc, vl_f1, vl_kap, vl_per = evaluate(model, test_loader)

        saved = ""
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), best_path)
            saved = " <- BEST"

        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f}(CE:{tr_ce:.3f}+KD:{tr_kd:.3f}) "
              f"TrF1:{tr_f1:.3f} ValAcc:{vl_acc:.3f} F1:{vl_f1:.3f} k:{vl_kap:.3f}{saved}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate(model, test_loader)

    print(f"\n  Seed {seed} FINAL: Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap, 'per_cls': fin_per})
    with open(csv_path, 'a', newline='') as f:
        csv.DictWriter(f, fields).writerow({
            "seed": seed, "acc": round(fin_acc, 4), "f1_macro": round(fin_f1, 4),
            "kappa": round(fin_kap, 4),
            "f1_Wake": round(fin_per[0], 4), "f1_N1": round(fin_per[1], 4),
            "f1_N2": round(fin_per[2], 4), "f1_N3": round(fin_per[3], 4),
            "f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs = np.array([r['acc'] for r in all_results]) * 100
f1s = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])
n1s = np.array([r['per_cls'][1] for r in all_results])

print(f"\n{'='*60}\nS5: CONTEXT-WINDOW STUDENT + KD -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")
print(f"N1 F1    : {n1s.mean():.4f} +- {n1s.std():.4f}")

print(f"\n{'='*60}\nFULL COMPARISON: SINGLE-EPOCH vs CONTEXT-WINDOW STUDENT\n{'='*60}")
print(f"{'Model':<50} {'Acc%':>7} {'F1':>7} {'N1':>7}")
print("-" * 74)
print(f"{'S1: Wearable baseline (no KD)':<50} {'57.09':>7} {'0.5112':>7} {'0.271':>7}")
print(f"{'S4(lambda=0.5): single-epoch + artifact + KD':<50} {'62.18':>7} {'0.5277':>7} {'0.246':>7}")
print(f"{'S5: CONTEXT-WINDOW + artifact + KD (this)':<50} {accs.mean():>7.2f} {f1s.mean():>7.4f} {n1s.mean():>7.3f}")
print(f"{'Teacher (PSG, reference)':<50} {'85.09':>7} {'0.8126':>7} {'---':>7}")
print("=" * 74)

print("\nInterpretation:")
print("  If N1 F1 improved substantially over S4 (0.246), the")
print("  single-epoch architecture WAS the bottleneck for N1, as")
print("  hypothesized -- this is a strong, mechanistically-grounded")
print("  finding for your thesis discussion. If overall F1 dropped")
print("  despite N1 improving, that's still a valid, reportable")
print("  trade-off to discuss (context helps N1 but may add noise")
print("  or overfitting risk elsewhere given the small dataset).")

print(f"\nSummary saved: {csv_path}")
print("Done!")

Device        : cuda
S5: CONTEXT-WINDOW student (+-7 epochs) + KD + artifact-weighted CE
KD lambda     : 0.5   Temperature: 3.0
Teacher       : MultiScaleSleepNetPlain, 5-seed ensemble (frozen)
Output        : D:\22\AA\evaluation\teacher-student\s5_context_window_kd
Split loaded -> Train:76  Test:20

Building context-window KD datasets (this uses more RAM than S1-S4)...
  Subjects loaded: 71  (large mismatches: 0)
  Samples: 66,763

  Subjects loaded: 19  (large mismatches: 0)
  Samples: 18,898
Datasets ready.

Loading 5 frozen teacher checkpoints...
Loaded 5 teacher models (frozen).

Class weights: {'Wake': np.float32(1.838), 'N1': np.float32(3.149), 'N2': np.float32(0.446), 'N3': np.float32(1.009), 'REM': np.float32(1.107)}

STUDENT SEED 42  (1/5)
  Student parameters: 238,469  (teacher frozen, not counted)
  Ep[01/30] Loss:4.010(CE:1.055+KD:5.911) TrF1:0.445 ValAcc:0.567 F1:0.475 k:0.363 <- BEST
  Ep[02/30] Loss:2.761(CE:0.844+KD:3.834) TrF1:0.598 ValAcc:0.618 F1:0.516 k:0.414 <- BE

In [1]:
# ============================================================
# S5: CONTEXT-WINDOW STUDENT + KD (lambda=0.5) +
#     ARTIFACT-WEIGHTED LOSS
#
# Motivation: S1-S4 all gave the student only a SINGLE center
# epoch of Zmax+E4 as input. The teacher, by contrast, sees a
# +-7 epoch context window (WINDOW=15). N1 is a transitional
# stage that is defined largely by surrounding-epoch context,
# so a single-epoch student is architecturally handicapped for
# N1 specifically -- consistent with N1 being the weakest class
# across every S1-S4 result so far.
#
# This script keeps the best loss configuration found so far
# (artifact-weighted CE + KD, lambda=0.5) and changes ONLY the
# input/architecture: each modality now receives a +-CONTEXT
# window of epochs, encoded per-epoch and then aggregated across
# the window with an inter-epoch BiGRU (mirroring the teacher's
# two-stage intra/inter-epoch design, but lightweight).
#
#   Zmax window (W,2,1920) -> per-epoch ZmaxEEGEncoder -> (W,d)
#                           -> inter-epoch BiGRU -> center repr
#   E4   window (W,3,1920) -> per-epoch E4Encoder      -> (W,d)
#                           -> inter-epoch BiGRU -> center repr
#   concat -> classifier
#
# Compare against S4(lambda=0.5) ensemble (single-epoch input,
# F1=0.5277 pre-ensemble) to isolate the effect of adding
# temporal context to the student.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
PSG_EEG_PATH      = r"D:\22\AA\preprocess\preprocessed_FFinal"
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"
TEACHER_CKPT_DIR  = r"D:\22\AA\evaluation\multiscale_plain_c7_88%"
EVAL_PATH         = r"D:\22\AA\evaluation\teacher-student\s5_context_window_kd_lamba=0.3"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
CONTEXT     = 7                    # same context size as the teacher, for direct comparability
WINDOW      = 2 * CONTEXT + 1       # 15
N_EPOCHS    = 30
BATCH_SIZE  = 32                    # reduced from 64: window inputs are ~15x larger per sample
SEEDS         = [42, 123, 256, 789, 999]
TEACHER_SEEDS = [42, 123, 256, 789, 999]

T_D_MODEL = 128
T_DROPOUT = 0.4
T_N_HEADS = 4

S_D_MODEL = 96
S_DROPOUT = 0.3

KD_LAMBDA      = 0.3     # best value found in the S2/S4 sweep
KD_TEMPERATURE = 3.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"S5: CONTEXT-WINDOW student (+-{CONTEXT} epochs) + KD + artifact-weighted CE")
print(f"KD lambda     : {KD_LAMBDA}   Temperature: {KD_TEMPERATURE}")
print(f"Teacher       : MultiScaleSleepNetPlain, 5-seed ensemble (frozen)")
print(f"Output        : {EVAL_PATH}")


# ============================================================
# ============  TEACHER ARCHITECTURE (frozen)  ================
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=T_D_MODEL, dropout=T_DROPOUT):
        super().__init__()
        mid = d_model // 4

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)
        self.medium = time_branch(50)
        self.large  = time_branch(100)

        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)
        feat = self.se(feat)
        return self.proj(feat)


class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=T_D_MODEL, n_heads=T_N_HEADS, dropout=T_DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            d_model, d_model // 2, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)
        attn_out = self.transformer(x)
        return self.norm2(x + attn_out)


class MultiScaleSleepNetPlain(nn.Module):
    """Teacher architecture -- identical to your trained checkpoints."""
    def __init__(self, in_ch=3, d_model=T_D_MODEL, n_layers=2,
                 dropout=T_DROPOUT, n_classes=5, context=CONTEXT):
        super().__init__()
        self.context = context
        self.d_model = d_model
        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)
        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.inter_pos = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes),
        )

    def forward(self, x):
        B, W, C, T = x.shape
        cnn_out = self.cnn(x.view(B * W, C, T)).permute(0, 2, 1)
        intra = self.intra_blocks(cnn_out)
        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)
        inter = self.inter_blocks(epoch_feat + self.inter_pos)
        center = inter[:, self.context, :]
        return self.classifier(center)


# ============================================================
# ========  STUDENT: PER-EPOCH ENCODER (same as S4)  ===========
# Applied to EACH epoch in the window via reshape, then an
# inter-epoch BiGRU aggregates across the window.
# ============================================================
class ZmaxEEGEncoder(nn.Module):
    """Per-epoch encoder -- unchanged from S4."""
    def __init__(self, in_ch=2, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        mid = d_model // 2

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2), nn.Dropout(dropout)
            )
        self.small, self.large = branch(15), branch(60)
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 1920)
            L_s, L_l = self.small(dummy).shape[2], self.large(dummy).shape[2]
        target_L = min(L_s, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.proj = nn.Sequential(nn.Conv1d(2 * mid, d_model, kernel_size=1),
                                   nn.BatchNorm1d(d_model), nn.GELU())
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (N, 2, 1920) where N = B*W (flattened batch*window)
        fs, fl = self.pool_s(self.small(x)), self.pool_l(self.large(x))
        feat = self.proj(torch.cat([fs, fl], dim=1)).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))   # (N, d_model)


class E4Encoder(nn.Module):
    """Per-epoch encoder -- unchanged from S4."""
    def __init__(self, in_ch=3, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, d_model // 2, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(d_model // 2), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model // 2, d_model, kernel_size=8, padding=4),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        feat = self.conv(x).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class InterEpochAggregator(nn.Module):
    """
    Aggregates a (B, W, d_model) sequence of per-epoch embeddings
    across the window using a BiGRU, then returns the CENTER
    epoch's contextualized representation -- this is what gives
    the student access to surrounding-epoch information, which
    single-epoch S1-S4 completely lacked.
    """
    def __init__(self, d_model=S_D_MODEL, context=CONTEXT):
        super().__init__()
        self.context = context
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, W, d_model)
        gru_out, _ = self.gru(x)              # (B, W, d_model)
        contextualized = self.norm(x + gru_out)
        return contextualized[:, self.context, :]   # (B, d_model) -- center epoch


class ContextStudentModel(nn.Module):
    """
    Context-window student: each modality gets a per-epoch
    encoder applied across the window, then an inter-epoch BiGRU
    aggregator extracts a context-aware representation of the
    CENTER epoch. Fusion is simple concat (best-performing so far).
    """
    def __init__(self, d_model=S_D_MODEL, n_classes=5, dropout=S_DROPOUT, context=CONTEXT):
        super().__init__()
        self.context = context
        self.zmax_enc = ZmaxEEGEncoder(in_ch=2, d_model=d_model, dropout=dropout)
        self.e4_enc   = E4Encoder(in_ch=3, d_model=d_model, dropout=dropout)
        self.zmax_agg = InterEpochAggregator(d_model=d_model, context=context)
        self.e4_agg   = InterEpochAggregator(d_model=d_model, context=context)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, zmax_x, e4_x):
        # zmax_x: (B, W, 2, 1920)   e4_x: (B, W, 3, 1920)
        B, W, Cz, Tz = zmax_x.shape
        _, _, Ce, Te = e4_x.shape

        z_flat = self.zmax_enc(zmax_x.reshape(B * W, Cz, Tz)).view(B, W, -1)
        e_flat = self.e4_enc(e4_x.reshape(B * W, Ce, Te)).view(B, W, -1)

        z_ctx = self.zmax_agg(z_flat)   # (B, d_model) -- center epoch, context-aware
        e_ctx = self.e4_agg(e_flat)     # (B, d_model)

        fused = torch.cat([z_ctx, e_ctx], dim=1)
        logits = self.classifier(fused)
        return logits


# ============================================================
# COMBINED DATASET (student now returns a WINDOW of Zmax+E4 epochs)
# ============================================================
class KDContextDataset(Dataset):
    def __init__(self, subject_list, psg_path, student_path, context=CONTEXT):
        self.context = context
        self.data = []
        self.index = []

        skipped_mismatch = 0
        for sub in subject_list:
            psg_fp = os.path.join(psg_path, f"{sub}.npz")
            stu_fp = os.path.join(student_path, f"{sub}.npz")
            if not (os.path.exists(psg_fp) and os.path.exists(stu_fp)):
                continue

            with np.load(psg_fp) as d:
                eeg = d['eeg'][:, [0, 1], :]
                eog = d['eog'][:, [0], :]
                psg_signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                psg_labels = d['labels'].copy()

            with np.load(stu_fp) as d:
                zmax_arr = d['zmax_eeg']
                e4_arr   = d['e4']
                stu_labels = d['labels'].copy()
                art_w = d['artifact_weight']

            n = min(len(psg_labels), len(stu_labels), zmax_arr.shape[0], e4_arr.shape[0])
            if abs(len(psg_labels) - len(stu_labels)) > 5:
                skipped_mismatch += 1

            sub_idx = len(self.data)
            self.data.append((psg_signal, psg_labels, zmax_arr, e4_arr, art_w, stu_labels, n))
            for i in range(n):
                self.index.append((sub_idx, i, n))

        print(f"  Subjects loaded: {len(self.data)}  (large mismatches: {skipped_mismatch})")
        print(f"  Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n = self.index[idx]
        psg_signal, psg_labels, zmax_arr, e4_arr, art_w, stu_labels, _ = self.data[sub_idx]

        # Teacher window (PSG)
        psg_window = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            psg_window.append(psg_signal[ei])
        psg_x = np.stack(psg_window, axis=0)

        # Student window (Zmax + E4) -- same padding-at-boundary convention
        zmax_window, e4_window = [], []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            zmax_window.append(zmax_arr[ei])
            e4_window.append(e4_arr[ei])
        zmax_x = np.stack(zmax_window, axis=0)   # (W, 2, 1920)
        e4_x   = np.stack(e4_window, axis=0)     # (W, 3, 1920)

        aw = float(art_w[center_i]) if center_i < len(art_w) else 1.0
        y  = int(stu_labels[center_i]) if center_i < len(stu_labels) else int(psg_labels[center_i])

        return (
            torch.FloatTensor(psg_x),
            torch.FloatTensor(zmax_x),
            torch.FloatTensor(e4_x),
            torch.tensor(aw, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
        )


# ============================================================
# LOAD SPLIT + BUILD DATASETS
# ============================================================
_train_path = os.path.join(PSG_EEG_PATH, "_train_subs.npy")
_test_path  = os.path.join(PSG_EEG_PATH, "_test_subs.npy")
TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()
print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Test:{len(TEST_SUBS)}")

print("\nBuilding context-window KD datasets (this uses more RAM than S1-S4)...")
train_ds = KDContextDataset(TRAIN_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print()
test_ds  = KDContextDataset(TEST_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print("Datasets ready.")


# ============================================================
# LOAD FROZEN TEACHER ENSEMBLE
# ============================================================
print(f"\nLoading {len(TEACHER_SEEDS)} frozen teacher checkpoints...")
teachers = []
for seed in TEACHER_SEEDS:
    ckpt = os.path.join(TEACHER_CKPT_DIR, f"best_seed{seed}.pt")
    if not os.path.exists(ckpt):
        print(f"  MISSING: {ckpt}")
        continue
    m = MultiScaleSleepNetPlain(in_ch=3).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    teachers.append(m)
print(f"Loaded {len(teachers)} teacher models (frozen).")


@torch.no_grad()
def teacher_soft_targets(psg_x, temperature=KD_TEMPERATURE):
    probs_sum = None
    for m in teachers:
        logits = m(psg_x)
        probs = F.softmax(logits / temperature, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(teachers)


# ============================================================
# CLASS WEIGHTS
# ============================================================
all_train_labels = []
for _, _, _, _, _, labs, _ in train_ds.data:
    all_train_labels.extend(labs.tolist())
label_counts = np.array([Counter(all_train_labels).get(i, 1) for i in range(5)], dtype=np.float32)
cw = torch.FloatTensor(label_counts.sum() / (5 * label_counts)).to(device)
print(f"\nClass weights: {dict(zip(LABEL_NAMES, cw.cpu().numpy().round(3)))}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def artifact_weighted_kd_loss_fn(student_logits, teacher_probs, labels, artifact_weight,
                                  temperature, lam, class_weights):
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    kd = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * (temperature ** 2)

    ce_per_sample = F.cross_entropy(student_logits, labels, weight=class_weights, reduction='none')
    weighted_ce = ce_per_sample * artifact_weight
    valid = artifact_weight > 0
    ce = weighted_ce[valid].mean() if valid.sum() > 0 else weighted_ce.mean()

    total = ce + lam * kd
    return total, kd.item(), ce.item()


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = total_kd = total_ce = 0
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, aw, y in loader:
        psg_x, zmax_x, e4_x, aw, y = (
            psg_x.to(device), zmax_x.to(device), e4_x.to(device),
            aw.to(device), y.to(device)
        )
        optimizer.zero_grad()
        teacher_probs = teacher_soft_targets(psg_x)
        student_logits = model(zmax_x, e4_x)
        loss, kd_val, ce_val = artifact_weighted_kd_loss_fn(
            student_logits, teacher_probs, y, aw, KD_TEMPERATURE, KD_LAMBDA, cw
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item(); total_kd += kd_val; total_ce += ce_val
        preds.extend(student_logits.argmax(1).cpu().numpy())
        labs_all.extend(y.cpu().numpy())

    n = len(loader)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    return total_loss / n, total_kd / n, total_ce / n, acc, f1


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, aw, y in loader:
        zmax_x, e4_x = zmax_x.to(device), e4_x.to(device)
        logits = model(zmax_x, e4_x)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs_all.extend(y.numpy())
    preds, labs_all = np.array(preds), np.array(labs_all)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs_all, preds)
    per_cls = f1_score(labs_all, preds, average=None, zero_division=0)
    return acc, f1, kappa, per_cls


# ============================================================
# TRAINING LOOP
# ============================================================
csv_path = os.path.join(EVAL_PATH, "summary.csv")
fields = ["seed", "acc", "f1_macro", "kappa", "f1_Wake", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fields).writeheader()

all_results = []
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSTUDENT SEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=0, generator=torch.Generator().manual_seed(seed))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = ContextStudentModel().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Student parameters: {n_params:,}  (teacher frozen, not counted)")

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4, betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_f1 = 0.0
    best_path = os.path.join(EVAL_PATH, f"best_student_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_kd, tr_ce, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler)
        vl_acc, vl_f1, vl_kap, vl_per = evaluate(model, test_loader)

        saved = ""
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), best_path)
            saved = " <- BEST"

        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f}(CE:{tr_ce:.3f}+KD:{tr_kd:.3f}) "
              f"TrF1:{tr_f1:.3f} ValAcc:{vl_acc:.3f} F1:{vl_f1:.3f} k:{vl_kap:.3f}{saved}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate(model, test_loader)

    print(f"\n  Seed {seed} FINAL: Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap, 'per_cls': fin_per})
    with open(csv_path, 'a', newline='') as f:
        csv.DictWriter(f, fields).writerow({
            "seed": seed, "acc": round(fin_acc, 4), "f1_macro": round(fin_f1, 4),
            "kappa": round(fin_kap, 4),
            "f1_Wake": round(fin_per[0], 4), "f1_N1": round(fin_per[1], 4),
            "f1_N2": round(fin_per[2], 4), "f1_N3": round(fin_per[3], 4),
            "f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs = np.array([r['acc'] for r in all_results]) * 100
f1s = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])
n1s = np.array([r['per_cls'][1] for r in all_results])

print(f"\n{'='*60}\nS5: CONTEXT-WINDOW STUDENT + KD -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")
print(f"N1 F1    : {n1s.mean():.4f} +- {n1s.std():.4f}")

print(f"\n{'='*60}\nFULL COMPARISON: SINGLE-EPOCH vs CONTEXT-WINDOW STUDENT\n{'='*60}")
print(f"{'Model':<50} {'Acc%':>7} {'F1':>7} {'N1':>7}")
print("-" * 74)
print(f"{'S1: Wearable baseline (no KD)':<50} {'57.09':>7} {'0.5112':>7} {'0.271':>7}")
print(f"{'S4(lambda=0.5): single-epoch + artifact + KD':<50} {'62.18':>7} {'0.5277':>7} {'0.246':>7}")
print(f"{'S5: CONTEXT-WINDOW + artifact + KD (this)':<50} {accs.mean():>7.2f} {f1s.mean():>7.4f} {n1s.mean():>7.3f}")
print(f"{'Teacher (PSG, reference)':<50} {'85.09':>7} {'0.8126':>7} {'---':>7}")
print("=" * 74)

print("\nInterpretation:")
print("  If N1 F1 improved substantially over S4 (0.246), the")
print("  single-epoch architecture WAS the bottleneck for N1, as")
print("  hypothesized -- this is a strong, mechanistically-grounded")
print("  finding for your thesis discussion. If overall F1 dropped")
print("  despite N1 improving, that's still a valid, reportable")
print("  trade-off to discuss (context helps N1 but may add noise")
print("  or overfitting risk elsewhere given the small dataset).")

print(f"\nSummary saved: {csv_path}")
print("Done!")

Device        : cuda
S5: CONTEXT-WINDOW student (+-7 epochs) + KD + artifact-weighted CE
KD lambda     : 0.3   Temperature: 3.0
Teacher       : MultiScaleSleepNetPlain, 5-seed ensemble (frozen)
Output        : D:\22\AA\evaluation\teacher-student\s5_context_window_kd_lamba=0.3
Split loaded -> Train:76  Test:20

Building context-window KD datasets (this uses more RAM than S1-S4)...
  Subjects loaded: 71  (large mismatches: 0)
  Samples: 66,763

  Subjects loaded: 19  (large mismatches: 0)
  Samples: 18,898
Datasets ready.

Loading 5 frozen teacher checkpoints...
Loaded 5 teacher models (frozen).

Class weights: {'Wake': np.float32(1.838), 'N1': np.float32(3.149), 'N2': np.float32(0.446), 'N3': np.float32(1.009), 'REM': np.float32(1.107)}

STUDENT SEED 42  (1/5)
  Student parameters: 238,469  (teacher frozen, not counted)
  Ep[01/30] Loss:2.837(CE:1.006+KD:6.104) TrF1:0.448 ValAcc:0.561 F1:0.475 k:0.359 <- BEST
  Ep[02/30] Loss:2.004(CE:0.800+KD:4.012) TrF1:0.595 ValAcc:0.610 F1:0.517 k:0